In [2]:
#Celda para pasar de Bronze a Silver
from pyspark.sql import functions as F, Window
import re

base = "Files/bronze/gtfs/"

def leer(nombre):
    df = spark.read.option("header", True).csv(base + nombre + ".txt")
    # 1. Nombres de columna limpios: sin espacios ni caracteres que Delta no admite
    nombres_limpios = [re.sub(r"[ ,;{}()\n\t=]", "", c.strip()) for c in df.columns]
    df = df.toDF(*nombres_limpios)
    # 2. Valores sin espacios sobrantes al principio y al final
    return df.select([F.trim(F.col(c)).alias(c) for c in df.columns])

for t in ["stops", "routes", "trips", "stop_times"]:
    (leer(t).write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable(f"silver_{t}"))
    print(f"silver_{t} guardada")

StatementMeta(, ea107845-e369-44b1-9c1e-20c4ad7091a2, 4, Finished, Available, Finished, False)

silver_stops guardada
silver_routes guardada
silver_trips guardada
silver_stop_times guardada


In [ ]:
#Celda de dimensiones
from pyspark.sql import functions as F, Window
import re

base = "Files/bronze/gtfs/"
stops = spark.table("silver_stops")
renfe = spark.table("silver_estaciones_renfe")  

dim_estacion = (stops
    .select("stop_id", "stop_name",
            F.col("stop_lat").cast("double").alias("lat"),
            F.col("stop_lon").cast("double").alias("lon"))
       .join(renfe.select(F.col("CODIGO").alias("stop_id"), F.col("PROVINCIA").alias("provincia")),
         "stop_id", "left"))
dim_estacion.write.mode("overwrite").format("delta").saveAsTable("gold_dim_estacion")

spark.table("silver_routes").select("route_id", "route_short_name", "route_long_name") \
    .write.mode("overwrite").format("delta").saveAsTable("gold_dim_servicio")

fechas = spark.sql("SELECT explode(sequence(to_date('2026-08-01'), to_date('2026-10-31'))) AS fecha")
dim_fecha = (fechas
    .withColumn("anio", F.year("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn("semana", F.weekofyear("fecha"))
    .withColumn("dia_semana", F.dayofweek("fecha"))
    .withColumn("es_finde", F.dayofweek("fecha").isin(1, 7)))
dim_fecha.write.mode("overwrite").format("delta").saveAsTable("gold_dim_fecha")

StatementMeta(, cef7a951-33c0-48dd-ad28-172cdc863cf3, 3, Finished, Available, Finished, False)

In [ ]:
display(spark.table("gold_dim_estacion")
    .groupBy(F.col("provincia").isNull().alias("sin_provincia"))
    .count())

StatementMeta(, 8b272786-3425-4741-94b0-7b2f74ef06e1, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bdbba6a3-ce73-4255-bcd5-aa1a2b395474)

In [ ]:
dim = spark.table("gold_dim_estacion")
print("filas:", dim.count(), "| stop_id distintos:", dim.select("stop_id").distinct().count())

StatementMeta(, 8b272786-3425-4741-94b0-7b2f74ef06e1, 8, Finished, Available, Finished, False)

filas: 1006 | stop_id distintos: 1006


In [ ]:
#Tabla de hechos simulada   
from pyspark.sql import functions as F, Window
import re
st = spark.table("silver_stop_times").withColumn("seq", F.col("stop_sequence").cast("int"))
w = Window.partitionBy("trip_id")
st = st.withColumn("min_seq", F.min("seq").over(w)).withColumn("max_seq", F.max("seq").over(w))

origen = st.filter("seq = min_seq").select("trip_id",
    F.col("stop_id").alias("estacion_origen"), F.col("departure_time").alias("hora_salida"))
destino = st.filter("seq = max_seq").select("trip_id",
    F.col("stop_id").alias("estacion_destino"), F.col("arrival_time").alias("hora_llegada"))

viajes = (spark.table("silver_trips").select("trip_id", "route_id")
    .join(origen, "trip_id").join(destino, "trip_id"))

causas = F.array(F.lit("Infraestructura"), F.lit("Operador"), F.lit("Externa"), F.lit("Otros operadores"))

fact = (viajes.crossJoin(fechas)
    .withColumn("r", F.rand(seed=7))
    .withColumn("retraso_min",
        F.when(F.col("r") < 0.70, F.floor(F.rand(1) * 5))
         .when(F.col("r") < 0.88, 5 + F.floor(F.rand(2) * 10))
         .otherwise(15 + F.floor(F.rand(3) * 60)).cast("int"))
    .withColumn("causa",
        F.when(F.col("retraso_min") < 5, F.lit("Sin incidencia"))
         .otherwise(F.element_at(causas, (F.floor(F.rand(4) * 4) + 1).cast("int"))))
    .withColumn("id_circulacion", F.monotonically_increasing_id())
    .drop("r"))

fact.write.mode("overwrite").format("delta").saveAsTable("gold_fact_circulacion")
display(fact.groupBy("causa").count())